# Тренажер: точка на поверхні конуса та її проекції

### Мета
Дослідити, як положення точки на поверхні конуса визначається його геометричними параметрами.

Конус задається:
- центром основи `(Xc, Yc, Zc)`;
- радіусом основи `R`;
- висотою `H`.

Положення точки задається:
- `φ` — кутом навколо осі;
- `Z` — висотою точки.

Радіус перерізу конуса на висоті `Z`:

**r(Z) = R · (1 − Z/H)**

Тому координати точки автоматично визначаються як:

**X = Xc + r(Z)·cos(φ)**

**Y = Yc + r(Z)·sin(φ)**

**Z = Z**

Система будує:
- 3D-модель конуса;
- точку A;
- A₁ — фронтальну проекцію;
- A₂ — горизонтальну проекцію;
- A₃ — профільну проекцію.

> **NO-CAD:** точка не малюється вручну. Вона отримується як результат параметричного опису поверхні.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Checkbox

def trainer(cx, cy, cz, radius, height, phi, point_z, show_links):
    # Не дозволяємо точці вийти за межі конуса
    point_z = min(point_z, height)

    angle = np.deg2rad(phi)

    # Радіус горизонтального перерізу конуса на висоті точки
    local_r = radius * (1 - point_z / height)

    # Координати точки A
    x = cx + local_r * np.cos(angle)
    y = cy + local_r * np.sin(angle)
    z = cz + point_z

    # Поверхня конуса
    t = np.linspace(0, 2*np.pi, 100)
    zz = np.linspace(cz, cz + height, 50)
    T, Z = np.meshgrid(t, zz)

    local_R = radius * (1 - (Z-cz)/height)
    X = cx + local_R*np.cos(T)
    Y = cy + local_R*np.sin(T)

    fig = plt.figure(figsize=(16, 9))

    axf = fig.add_subplot(221)
    axp = fig.add_subplot(222)
    axh = fig.add_subplot(223)
    ax3 = fig.add_subplot(224, projection="3d")

    # ---------- ФРОНТАЛЬНА ПРОЕКЦІЯ ----------
    axf.set_title("ФРОНТАЛЬНА ПРОЕКЦІЯ XZ", fontweight="bold")
    axf.set_xlabel("X")
    axf.set_ylabel("Z")
    axf.grid(alpha=.25)
    axf.set_aspect("equal", adjustable="box")

    axf.plot(
        [cx-radius, cx, cx+radius],
        [cz, cz+height, cz],
        linewidth=1.8
    )
    axf.plot(
        [cx-radius, cx+radius],
        [cz, cz],
        linewidth=1.8
    )

    axf.scatter(x, z, s=100)
    axf.annotate("A₁", (x,z), xytext=(8,8),
                 textcoords="offset points", fontsize=13,
                 fontweight="bold")

    # ---------- ПРОФІЛЬНА ПРОЕКЦІЯ ----------
    axp.set_title("ПРОФІЛЬНА ПРОЕКЦІЯ YZ", fontweight="bold")
    axp.set_xlabel("Y")
    axp.set_ylabel("Z")
    axp.grid(alpha=.25)
    axp.set_aspect("equal", adjustable="box")

    axp.plot(
        [cy-radius, cy, cy+radius],
        [cz, cz+height, cz],
        linewidth=1.8
    )
    axp.plot(
        [cy-radius, cy+radius],
        [cz, cz],
        linewidth=1.8
    )

    axp.scatter(y, z, s=100)
    axp.annotate("A₃", (y,z), xytext=(8,8),
                 textcoords="offset points", fontsize=13,
                 fontweight="bold")

    # ---------- ГОРИЗОНТАЛЬНА ПРОЕКЦІЯ ----------
    axh.set_title("ГОРИЗОНТАЛЬНА ПРОЕКЦІЯ XY", fontweight="bold")
    axh.set_xlabel("X")
    axh.set_ylabel("Y")
    axh.grid(alpha=.25)
    axh.set_aspect("equal", adjustable="box")

    circle_x = cx + radius*np.cos(t)
    circle_y = cy + radius*np.sin(t)

    axh.plot(circle_x, circle_y, linewidth=1.8)
    axh.fill(circle_x, circle_y, alpha=.08)

    # Коло, що проходить через точку A
    sec_x = cx + local_r*np.cos(t)
    sec_y = cy + local_r*np.sin(t)
    axh.plot(sec_x, sec_y, linestyle="--", linewidth=1.0, alpha=.6)

    axh.scatter(x, y, s=100)
    axh.annotate("A₂", (x,y), xytext=(8,8),
                 textcoords="offset points", fontsize=13,
                 fontweight="bold")

    # ---------- ПРОЕКЦІЙНІ ЗВ'ЯЗКИ ----------
    if show_links:
        axf.axvline(x, linestyle="--", linewidth=1, alpha=.35)
        axf.axhline(z, linestyle="--", linewidth=1, alpha=.35)

        axh.axvline(x, linestyle="--", linewidth=1, alpha=.35)
        axh.axhline(y, linestyle="--", linewidth=1, alpha=.35)

        axp.axvline(y, linestyle="--", linewidth=1, alpha=.35)
        axp.axhline(z, linestyle="--", linewidth=1, alpha=.35)

    # ---------- 3D ----------
    ax3.set_title("3D-МОДЕЛЬ: ТОЧКА НА ПОВЕРХНІ КОНУСА",
                  fontweight="bold")
    ax3.plot_surface(X, Y, Z, alpha=.22, linewidth=0)

    # Основа
    ax3.plot(
        circle_x, circle_y, np.full_like(t, cz),
        linewidth=1.6
    )

    # Вершина
    ax3.scatter(cx, cy, cz+height, s=55)

    # Вісь
    ax3.plot(
        [cx,cx], [cy,cy], [cz,cz+height],
        linestyle="--", linewidth=1.2, alpha=.6
    )

    # Точка
    ax3.scatter(x, y, z, s=130)
    ax3.text(x, y, z, "  A", fontsize=13, fontweight="bold")

    # Радіус до точки
    ax3.plot(
        [cx,x], [cy,y], [z,z],
        linestyle="--", linewidth=1.2
    )

    # Вертикальний зв'язок
    ax3.plot(
        [x,x], [y,y], [cz,z],
        linestyle=":", linewidth=1.2
    )

    ax3.set_xlabel("X")
    ax3.set_ylabel("Y")
    ax3.set_zlabel("Z")

    span = max(2*radius, height, 4)
    ax3.set_xlim(cx-span/2, cx+span/2)
    ax3.set_ylim(cy-span/2, cy+span/2)
    ax3.set_zlim(cz, cz+span)

    fig.suptitle(
        f"A({x:.2f}, {y:.2f}, {z:.2f})   •   "
        f"φ = {phi:.0f}°   •   r = {local_r:.2f}   •   "
        f"R = {radius:.1f}   •   H = {height:.1f}",
        fontsize=15, fontweight="bold"
    )

    plt.tight_layout()
    plt.show()

interact(
    trainer,
    cx=FloatSlider(min=0, max=10, step=1, value=5, description="Центр X"),
    cy=FloatSlider(min=0, max=10, step=1, value=5, description="Центр Y"),
    cz=FloatSlider(min=0, max=5, step=1, value=0, description="Центр Z"),
    radius=FloatSlider(min=1, max=6, step=1, value=4, description="Радіус R"),
    height=FloatSlider(min=2, max=10, step=1, value=7, description="Висота H"),
    phi=FloatSlider(min=0, max=360, step=5, value=45, description="Кут φ"),
    point_z=FloatSlider(min=0, max=10, step=.5, value=3.5, description="Висота точки"),
    show_links=Checkbox(value=True, description="Показувати зв'язки")
)

## Завдання для студента

### 1. Базовий випадок
Встановіть:

- `R = 4`
- `H = 8`
- `φ = 0°`
- `Z точки = 4`

Обчисліть координати точки A вручну та порівняйте з Notebook.

### 2. Зміна висоти
Не змінюючи `φ`, рухайте точку від основи до вершини.

Спостерігайте, як змінюється її відстань від осі.

### 3. Зміна кута
Зафіксуйте висоту та змінюйте `φ`.

Як змінюються X та Y?

### 4. Проекції
Для вибраного положення запишіть:

**A(X,Y,Z)**

**A₁(X,Z)**

**A₂(X,Y)**

**A₃(Y,Z)**

### Головний висновок

У конуса радіус положення точки залежить від її висоти:

**зміна Z → зміна локального радіуса → зміна X та Y → зміна трьох проекцій.**

Це приклад параметричного опису складнішої просторової поверхні.